
# Modelos de Machine Learning - Saber 11

## Proyecto final de Ciencia de Datos

### Objetivo

Desarrollar y comparar tres modelos de Machine Learning
para predecir el puntaje global de los estudiantes en
las pruebas Saber 11, utilizando características
socioeconómicas, familiares y educativas.

### Tipo de problema

Regresión supervisada.

### Modelos seleccionados

1. Regresión lineal.
2. Árbol de decisión.
3. Random Forest.

### Metodología

- Carga y revisión del dataset preparado.
- Separación de variables predictoras y variable objetivo.
- División de datos en entrenamiento y prueba.
- Preprocesamiento de variables categóricas y valores faltantes.
- Entrenamiento de tres modelos.
- Ajuste de hiperparámetros.
- Evaluación mediante MAE, RMSE y R².
- Comparación y selección del mejor modelo.
- Exportación del modelo para integrarlo con Streamlit.


In [1]:

import pandas as pd

# Cargar el dataset directamente desde GitHub
url = "https://raw.githubusercontent.com/helenx00/Proyecto_db/helen/data/saber11_modelado.csv"

df = pd.read_csv(url)

# Comprobar los datos
print("Dimensiones del dataset:", df.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nPrimeras cinco filas:")
display(df.head())


Dimensiones del dataset: (10000, 9)

Columnas:
['fami_estratovivienda', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_tieneinternet', 'fami_tienecomputador', 'cole_naturaleza', 'cole_area_ubicacion', 'estu_genero', 'punt_global']

Primeras cinco filas:


,fami_estratovivienda,fami_educacionmadre,fami_educacionpadre,fami_tieneinternet,fami_tienecomputador,cole_naturaleza,cole_area_ubicacion,estu_genero,punt_global
0,Estrato 2,Educación profesional completa,Técnica o tecnológica completa,Si,Si,OFICIAL,RURAL,F,244
1,Estrato 3,Educación profesional completa,Secundaria (Bachillerato) completa,Si,No,OFICIAL,RURAL,M,238
2,Estrato 1,Secundaria (Bachillerato) incompleta,Primaria incompleta,No,No,OFICIAL,RURAL,F,325
3,Sin Estrato,Educación profesional completa,Secundaria (Bachillerato) incompleta,Si,No,OFICIAL,RURAL,M,238
4,Estrato 5,Educación profesional completa,Secundaria (Bachillerato) completa,Si,Si,OFICIAL,URBANO,F,202



## 2. Preparación de los datos

Se separan las ocho variables predictoras (X) de la
variable objetivo (y), correspondiente al puntaje global
de Saber 11.

El conjunto de datos se divide en un 80 % para
entrenamiento y un 20 % para prueba, utilizando una
semilla aleatoria de 42 para garantizar la
reproducibilidad de los resultados.

Esta separación permite evaluar el rendimiento de
los modelos utilizando datos que no participaron
en su entrenamiento.


In [2]:

from sklearn.model_selection import train_test_split

# Variables predictoras
X = df.drop(columns=["punt_global"])

# Variable objetivo
y = df["punt_global"]

# Separación de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Registros de entrenamiento:", len(X_train))
print("Registros de prueba:", len(X_test))
print("Variables predictoras:", X.shape[1])


Registros de entrenamiento: 8000
Registros de prueba: 2000
Variables predictoras: 8



## 3. Preprocesamiento de variables

Las ocho variables predictoras son categóricas, por lo
que requieren una transformación antes de entrenar
los modelos.

Se utiliza SimpleImputer con la estrategia most_frequent
para reemplazar los valores faltantes por la categoría
más frecuente.

Posteriormente, se aplica OneHotEncoder para convertir
las categorías en variables numéricas binarias.

Estas transformaciones se incorporarán a pipelines
de entrenamiento para evitar fugas de información
entre los conjuntos de entrenamiento y prueba.


In [3]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Identificar variables categóricas
columnas_categoricas = X.columns.tolist()

# Crear pipeline de preprocesamiento
transformador_categorico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("codificador", OneHotEncoder(handle_unknown="ignore"))
])

# Aplicar transformaciones a las 8 variables
preprocesador = ColumnTransformer(
    transformers=[
        ("categoricas", transformador_categorico, columnas_categoricas)
    ]
)

print("Preprocesador creado correctamente.")
print("Variables categóricas:", len(columnas_categoricas))


Preprocesador creado correctamente.
Variables categóricas: 8



## 4. Entrenamiento de modelos de Machine Learning

### 4.1. Modelo 1: Regresión lineal

La regresión lineal es un algoritmo de aprendizaje
supervisado que busca establecer una relación entre
las variables predictoras y una variable objetivo
numérica.

En este proyecto se utiliza para estimar el puntaje
global de las pruebas Saber 11 a partir de ocho
características socioeconómicas, familiares y educativas.

Se emplea como modelo de referencia para comparar
el rendimiento de algoritmos más complejos.


In [4]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import numpy as np

# Crear pipeline con preprocesamiento y regresión lineal
modelo_lineal = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", LinearRegression())
])

# Entrenar el modelo
modelo_lineal.fit(X_train, y_train)

# Realizar predicciones
pred_lineal = modelo_lineal.predict(X_test)

# Calcular métricas
mae_lineal = mean_absolute_error(y_test, pred_lineal)
rmse_lineal = np.sqrt(mean_squared_error(y_test, pred_lineal))
r2_lineal = r2_score(y_test, pred_lineal)

print("RESULTADOS - REGRESIÓN LINEAL")
print(f"MAE: {mae_lineal:.2f}")
print(f"RMSE: {rmse_lineal:.2f}")
print(f"R²: {r2_lineal:.4f}")


RESULTADOS - REGRESIÓN LINEAL
MAE: 34.49
RMSE: 43.11
R²: 0.1740



### 4.2. Modelo 2: Árbol de decisión

El árbol de decisión es un algoritmo de aprendizaje
supervisado que realiza predicciones mediante reglas
y divisiones sucesivas de los datos.

A diferencia de la regresión lineal, puede identificar
relaciones no lineales entre las características
socioeconómicas, familiares y educativas y el puntaje
global de Saber 11.

Se entrena inicialmente con una profundidad máxima
de 8 niveles para limitar su complejidad y reducir
el riesgo de sobreajuste.


In [5]:

from sklearn.tree import DecisionTreeRegressor

# Crear pipeline con árbol de decisión
modelo_arbol = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ))
])

# Entrenar el modelo
modelo_arbol.fit(X_train, y_train)

# Realizar predicciones
pred_arbol = modelo_arbol.predict(X_test)

# Calcular métricas
mae_arbol = mean_absolute_error(y_test, pred_arbol)
rmse_arbol = np.sqrt(mean_squared_error(y_test, pred_arbol))
r2_arbol = r2_score(y_test, pred_arbol)

print("RESULTADOS - ÁRBOL DE DECISIÓN")
print(f"MAE: {mae_arbol:.2f}")
print(f"RMSE: {rmse_arbol:.2f}")
print(f"R²: {r2_arbol:.4f}")


RESULTADOS - ÁRBOL DE DECISIÓN
MAE: 35.03
RMSE: 43.81
R²: 0.1471



### 4.3. Modelo 3: Random Forest

Random Forest es un algoritmo de aprendizaje supervisado
que combina múltiples árboles de decisión para realizar
predicciones.

Este modelo permite identificar relaciones no lineales
entre las variables socioeconómicas, familiares y
educativas y el puntaje global de Saber 11.

Se utilizan inicialmente 100 árboles, una profundidad
máxima de 10 niveles y una semilla aleatoria de 42.

Su rendimiento se evaluará mediante MAE, RMSE y R²
para compararlo con los modelos anteriores.


In [6]:

from sklearn.ensemble import RandomForestRegressor

# Crear pipeline con Random Forest
modelo_forest = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

# Entrenar el modelo
modelo_forest.fit(X_train, y_train)

# Realizar predicciones
pred_forest = modelo_forest.predict(X_test)

# Calcular métricas
mae_forest = mean_absolute_error(y_test, pred_forest)
rmse_forest = np.sqrt(mean_squared_error(y_test, pred_forest))
r2_forest = r2_score(y_test, pred_forest)

print("RESULTADOS - RANDOM FOREST")
print(f"MAE: {mae_forest:.2f}")
print(f"RMSE: {rmse_forest:.2f}")
print(f"R²: {r2_forest:.4f}")


RESULTADOS - RANDOM FOREST
MAE: 34.54
RMSE: 43.07
R²: 0.1755



## 5. Optimización de hiperparámetros

Se realiza un ajuste de hiperparámetros para los modelos
Árbol de decisión y Random Forest utilizando
RandomizedSearchCV.

Se emplea validación cruzada de tres particiones
y se seleccionan las configuraciones que minimizan
el error cuadrático medio (RMSE).

El ajuste se realiza exclusivamente sobre los datos
de entrenamiento, conservando el conjunto de prueba
para la evaluación final.


In [7]:

from sklearn.model_selection import RandomizedSearchCV

# Configuraciones para el árbol de decisión
parametros_arbol = {
    "modelo__max_depth": [3, 5, 8, 10, 15, None],
    "modelo__min_samples_split": [2, 5, 10, 20],
    "modelo__min_samples_leaf": [1, 2, 5, 10]
}

busqueda_arbol = RandomizedSearchCV(
    estimator=modelo_arbol,
    param_distributions=parametros_arbol,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1
)

busqueda_arbol.fit(X_train, y_train)

print("MEJORES PARÁMETROS - ÁRBOL")
print(busqueda_arbol.best_params_)
print("RMSE promedio en validación cruzada:",
      round(-busqueda_arbol.best_score_, 2))


MEJORES PARÁMETROS - ÁRBOL
{'modelo__min_samples_split': 10, 'modelo__min_samples_leaf': 5, 'modelo__max_depth': 8}
RMSE promedio en validación cruzada: 43.29



### 5.2. Optimización de Random Forest

Se utiliza RandomizedSearchCV con validación cruzada
de tres particiones para identificar una configuración
adecuada del modelo Random Forest.

Se ajustan el número de árboles, la profundidad máxima,
el número mínimo de muestras para dividir un nodo y
el número mínimo de muestras por hoja.

La selección se realiza minimizando el RMSE promedio
en validación cruzada sobre los datos de entrenamiento.


In [8]:

# Configuraciones para Random Forest
parametros_forest = {
    "modelo__n_estimators": [50, 100, 150],
    "modelo__max_depth": [5, 8, 10, 15, None],
    "modelo__min_samples_split": [2, 5, 10],
    "modelo__min_samples_leaf": [1, 2, 5]
}

busqueda_forest = RandomizedSearchCV(
    estimator=modelo_forest,
    param_distributions=parametros_forest,
    n_iter=8,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1
)

busqueda_forest.fit(X_train, y_train)

print("MEJORES PARÁMETROS - RANDOM FOREST")
print(busqueda_forest.best_params_)
print("RMSE promedio en validación cruzada:",
      round(-busqueda_forest.best_score_, 2))


MEJORES PARÁMETROS - RANDOM FOREST
{'modelo__n_estimators': 50, 'modelo__min_samples_split': 10, 'modelo__min_samples_leaf': 2, 'modelo__max_depth': 8}
RMSE promedio en validación cruzada: 42.36



## 6. Evaluación y comparación de modelos

Se comparan tres algoritmos de regresión:

1. Regresión lineal.
2. Árbol de decisión optimizado.
3. Random Forest optimizado.

Para garantizar una comparación consistente, todos
los modelos se evalúan utilizando el mismo conjunto
de prueba, compuesto por 2.000 registros.

Se emplean tres métricas:

- MAE: error absoluto medio.
- RMSE: raíz del error cuadrático medio.
- R²: coeficiente de determinación.

Los valores menores de MAE y RMSE indican mejores
predicciones, mientras que un R² mayor indica una
mejor capacidad explicativa.


In [9]:

# Recuperar los mejores modelos optimizados
modelo_arbol_opt = busqueda_arbol.best_estimator_
modelo_forest_opt = busqueda_forest.best_estimator_

# Modelos que vamos a comparar
modelos_finales = {
    "Regresión lineal": modelo_lineal,
    "Árbol de decisión optimizado": modelo_arbol_opt,
    "Random Forest optimizado": modelo_forest_opt
}

resultados = []

for nombre, modelo in modelos_finales.items():
    predicciones = modelo.predict(X_test)

    resultados.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test, predicciones),
        "RMSE": np.sqrt(mean_squared_error(y_test, predicciones)),
        "R²": r2_score(y_test, predicciones)
    })

# Crear tabla comparativa
tabla_resultados = pd.DataFrame(resultados)

# Ordenar de menor a mayor RMSE
tabla_resultados = tabla_resultados.sort_values(
    by="RMSE"
).reset_index(drop=True)

print("COMPARACIÓN FINAL DE MODELOS")
display(tabla_resultados.round(4))

print("\nMEJOR MODELO SEGÚN RMSE:")
print(tabla_resultados.iloc[0]["Modelo"])


COMPARACIÓN FINAL DE MODELOS


,Modelo,MAE,RMSE,R²
0,Random Forest optimizado,34.5366,43.0518,0.1762
1,Regresión lineal,34.4945,43.1097,0.1740
2,Árbol de decisión optimizado,35.2354,43.9658,0.1408



MEJOR MODELO SEGÚN RMSE:
Random Forest optimizado


In [ ]:
# Visualización comparativa de los errores RMSE calculados
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
barras = plt.bar(tabla_resultados["Modelo"], tabla_resultados["RMSE"], color=["#4378A7", "#79A7C8", "#B2C9DA"])
plt.title("Comparación del RMSE de los modelos")
plt.ylabel("RMSE (puntos Saber 11)")
plt.xlabel("Modelo evaluado")
plt.ylim(0, max(tabla_resultados["RMSE"]) * 1.15)
plt.xticks(rotation=12, ha="right")
for barra in barras:
    valor = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2, valor + 0.3, f"{valor:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()



## 7. Selección del mejor modelo y conclusiones

### Selección del modelo

Después de entrenar y evaluar tres algoritmos de
regresión, se seleccionó Random Forest optimizado
como modelo final, debido a que obtuvo el menor RMSE
(43.0518) y el mayor R² (0.1762) sobre el conjunto
de prueba.

Su configuración óptima fue:

- Número de árboles: 50.
- Profundidad máxima: 8.
- Muestras mínimas para dividir un nodo: 10.
- Muestras mínimas por hoja: 2.

Aunque la regresión lineal obtuvo un MAE ligeramente
menor (34.4945 frente a 34.5366), Random Forest
presentó el mejor resultado según el criterio
principal de selección: RMSE.

### Conclusiones

1. Se desarrollaron tres modelos de Machine Learning
para estimar el puntaje global de Saber 11 a partir
de ocho variables socioeconómicas, familiares y
educativas.

2. Random Forest optimizado obtuvo el menor RMSE
(43.0518), aunque su ventaja sobre la regresión
lineal fue pequeña.

3. El R² de 0.1762 indica que el modelo explica
aproximadamente el 17.62 % de la variación del
puntaje global, por lo que su capacidad predictiva
es limitada.

4. Los resultados sugieren que las variables
seleccionadas aportan información para estimar el
puntaje, pero no son suficientes para explicar
la mayor parte de sus diferencias.

5. El análisis no permite establecer relaciones
causales entre las características familiares,
socioeconómicas y el rendimiento académico.

6. Como se trabajó con los primeros 10.000 registros
del conjunto de datos, la muestra no necesariamente
es representativa de todos los estudiantes
colombianos. Por ello, las conclusiones deben
interpretarse dentro de las limitaciones del estudio.



## 8. Exportación del modelo final

Se guarda el modelo Random Forest optimizado mediante
joblib para utilizarlo posteriormente en Streamlit
y realizar predicciones con nuevos datos.


In [ ]:
import joblib
from pathlib import Path

# Guardar el pipeline completo: preprocesamiento + Random Forest optimizado.
# En Colab se guarda en el directorio de trabajo actual.
ruta_modelo = Path("modelo_saber11.joblib")
joblib.dump(modelo_forest_opt, ruta_modelo)
print(f"Modelo exportado correctamente en: {ruta_modelo.resolve()}")
